In [59]:
import arcpy
import os
from pathlib import Path
from matplotlib import pyplot as pl
import pandas as pd

In [2]:
cwd = Path(os.getcwd())
arcpy.env.workspace = cwd
arcpy.env.overwriteOutput = True

# Question 2

In [4]:
# Adding a 20 mile buffer around Alaska
alaskaFolder = cwd/"Earthquakes_and_Alaska"/"Alaska"
alaskaShapefile = alaskaFolder/"Alaska.shp"
alaskaBufferPath = alaskaFolder/"Alaska_50mileBuffer.shp"
arcpy.analysis.Buffer(alaskaShapefile, alaskaBufferPath, "20 Miles")

<Result 'C:\\Users\\brian\\Documents\\2026-2027\\Fall 2026\\Spatial Programming\\Week 10\\Earthquakes_and_Alaska\\Alaska\\Alaska_50mileBuffer.shp'>

In [6]:
# Clipping the 2 earthquake datasets to be within Alaska or within 20 miles of Alaska
earthquake1folder = cwd/"Earthquakes_and_Alaska"/"earthquake_data_1"
earthquakeData1 = earthquake1folder/"earthquake_data_1.shp"
earthquake1clipped = earthquake1folder/"AlaskaEarthquake1.shp"
arcpy.analysis.Clip(earthquakeData1, alaskaBufferPath, earthquake1clipped)

earthquake2folder = cwd/"Earthquakes_and_Alaska"/"earthquake_data_2"
earthquakeData2 = earthquake2folder/"earthquake_data_2.shp"
earthquake2clipped = earthquake2folder/"AlaskaEarthquake2.shp"
arcpy.analysis.Clip(earthquakeData2, alaskaBufferPath, earthquake2clipped)

<Result 'C:\\Users\\brian\\Documents\\2026-2027\\Fall 2026\\Spatial Programming\\Week 10\\Earthquakes_and_Alaska\\earthquake_data_2\\AlaskaEarthquake2.shp'>

Spatial join of the 2 clipped earthquake shapefiles

In [9]:
# Specifying fields to keep
alaskaEarthquakesFieldMappings = arcpy.FieldMappings()
alaskaEarthquakesFieldMappings.addTable(earthquake1clipped)
alaskaEarthquakesFieldMappings.addTable(earthquake2clipped)
relevantFields = ["QUAKE_ID", "YEAR", "LOCATION", "LONGITUDE", "LATITUDE", "DEPTH_KM", "MMIO", "FELT_AREA"]
for field in alaskaEarthquakesFieldMappings.fields:
    fieldName = field.name
    if fieldName not in relevantFields:
        fieldIndex = alaskaEarthquakesFieldMappings.findFieldMapIndex(fieldName)
        alaskaEarthquakesFieldMappings.removeFieldMap(fieldIndex)

The 2 clipped earthquake shapefiles each contain 911 earthquakes as points in identical locations. Both files contain identical earthquakes, it's just that each contain slightly different fields. Doing the spatial join now will combine them together into a single dataset of the 911 earthquakes.

In [10]:
combinedAlaskaEarthquakes = cwd/"Earthquakes_and_Alaska"/"AlaskaEarthquakesSpatialJoin.shp"
arcpy.analysis.SpatialJoin(earthquake1clipped, earthquake2clipped, combinedAlaskaEarthquakes, "", "", alaskaEarthquakesFieldMappings, "ARE_IDENTICAL_TO")

<Result 'C:\\Users\\brian\\Documents\\2026-2027\\Fall 2026\\Spatial Programming\\Week 10\\Earthquakes_and_Alaska\\AlaskaEarthquakesSpatialJoin.shp'>

# Question 3

In [39]:
cursor = arcpy.da.SearchCursor(combinedAlaskaEarthquakes, ["YEAR", "MMIO"], "YEAR >= 1900 AND YEAR <= 1999 AND MMIO <> ' '")

In [40]:
years = []
mmioValues = []
for row in cursor:
    years.append(row[0])
    mmioValues.append(int(row[1]))

At this point, I'm having a lot of trouble trying to display the histograms on Jupyter Notebook. I will export the 2 lists as a .csv file below and write a separate Python script to plot the data.

In [63]:
alaskaEarthquakeDataPath = cwd/"Earthquakes_and_Alaska"/"Year and intensity of each earthquake.csv"
alaskaEarthquakeData = pd.DataFrame({"year": years, "intensity": mmioValues})
if os.path.exists(alaskaEarthquakeDataPath) is False:
    alaskaEarthquakeData.to_csv(alaskaEarthquakeDataPath, index=False)

# Question 4

In [67]:
updateCursor = arcpy.da.UpdateCursor(combinedAlaskaEarthquakes, ["DEPTH_KM"])
shallowEarthquakes = []
earthquakesWithData = []
for row in updateCursor:
    depth = row[0]
    if depth == -9999:
        updateCursor.deleteRow()
    else:
        earthquakesWithData.append(depth)
        if depth < 5:
            shallowEarthquakes.append(depth)
numberOfEarthquakes = len(earthquakesWithData)
meanDepth = sum(earthquakesWithData)/numberOfEarthquakes
print("There were a total of {0} earthquakes with depth data, and the mean depth is {1:.2f} km".format(numberOfEarthquakes, meanDepth))

There were a total of 673 earthquakes with depth data, and the mean depth is 37.82 km
